# Mini-project: student performance 🎓

You finished the 18 videos. Now prove it to yourself.

**Files** (put all three next to this notebook):
- `students_project.csv`: 1000 students, but **messy** on purpose
- `students_extra.json`: extra info (attendance, study hours) for most students

**Rules**
1. Do each task in the empty cell under it.
2. Compare your result with the **✅ checkpoint**. If it matches, move on. If not, debug before continuing; later parts depend on earlier ones.
3. Open `mini_project_solutions.ipynb` **only** after you've tried.
4. If something gets weird, **Restart Kernel + Run All** (remember the all-75s bug 😄).

Rough time: 60–90 min.

Setup: run this cell.

In [1]:
import pandas as pd

---
## Part 1: load and discover

**1a.** Read `students_project.csv` into `df`. Show the first 5 rows, the shape, and `info()`.

✅ Checkpoint: shape is `(1012, 9)`. Look at the `Writing Score` dtype in `info()`. Is it a number column? 🤔

In [2]:
df=pd.read_csv('students_project.csv')
display(df.info())
display(df.head())
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 1012 entries, 0 to 1011
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Student ID                   1012 non-null   int64  
 1   Gender                       1012 non-null   str    
 2   race/ethnicity               1012 non-null   str    
 3   parental level of education  1012 non-null   str    
 4   lunch                        1012 non-null   str    
 5   test preparation course      1012 non-null   str    
 6   Math Score                   994 non-null    float64
 7   Reading Score                1003 non-null   float64
 8   Writing Score                1012 non-null   str    
dtypes: float64(2), int64(1), str(6)
memory usage: 71.3 KB


None

,Student ID,Gender,race/ethnicity,parental level of education,lunch,test preparation course,Math Score,Reading Score,Writing Score
0,1821,female,group A,some high school,standard,completed,85.0,90.0,92
1,1734,male,group D,some high school,standard,none,55.0,47.0,44
2,1829,female,group D,some high school,free/reduced,completed,69.0,86.0,81
3,1570,male,group D,bachelor's degree,free/reduced,none,68.0,68.0,67
4,1902,female,group C,master's degree,standard,none,73.0,78.0,74


(1012, 9)

**1b.** Find the mess:
- How many unique values does `Gender` have? Show its `value_counts()`.
- How many missing values are there in the whole table?
- How many duplicate rows are there?

✅ Checkpoint: `10` unique gender values (there should only be 2!), `27` missing values, `12` duplicate rows.

In [3]:
display(df['Gender'].nunique())
display(df['Gender'].value_counts())
display(df.isnull().sum().sum())
display(df.duplicated().sum())

10

Gender
female     502
male       470
FEMALE       7
MALE         6
female       6
 female      6
Female       4
 male        4
Male         4
male         3
Name: count, dtype: int64

np.int64(27)

np.int64(12)

---
## Part 2: clean

**2a.** Clean the column names: lowercase, strip spaces, and replace spaces and `/` with `_`.

💡 Hint: `df.columns.str.lower()`, `.str.replace(...)`

✅ Checkpoint: `['student_id', 'gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score']`

In [4]:
df.columns= (df.columns.str.strip()
                       .str.lower()
                       .str.replace(' ','_')
                       .str.replace('/','_'))
df.columns.tolist()

['student_id',
 'gender',
 'race_ethnicity',
 'parental_level_of_education',
 'lunch',
 'test_preparation_course',
 'math_score',
 'reading_score',
 'writing_score']

**2b.** Drop the duplicate rows.

✅ Checkpoint: shape is `(1000, 9)`.

In [5]:
df=df.drop_duplicates()
df.shape

(1000, 9)

**2c.** Fix `gender` so every value is just `female` or `male`.

✅ Checkpoint: `value_counts()` gives `female 518`, `male 482`.

In [6]:
df['gender']=df['gender'].str.strip().str.lower()
df['gender'].value_counts()

gender
female    518
male      482
Name: count, dtype: int64

**2d.** `writing_score` has text in it (like `"absent"`). Convert it to numbers, and turn the text into `NaN`.

💡 Hint: `pd.to_numeric(..., errors='coerce')`

✅ Checkpoint: `writing_score` now has `5` NaN values.

In [7]:
df['writing_score']=pd.to_numeric(df['writing_score'],errors='coerce')
df['writing_score'].isna().sum()

np.int64(5)

**2e.** Scores must be between 0 and 100. Some are typos (150, 999, -5…). Replace anything **outside** 0–100 with `NaN` in all three score columns.

💡 Hint: `.between(0, 100)` + `.where(...)`. `where` keeps values where the condition is True and replaces the rest.

✅ Checkpoint: NaN counts are math `22`, reading `11`, writing `5`.

In [8]:
scores =['math_score','reading_score','writing_score']
for col in scores:
    df[col]=df[col].where(df[col].between(0,100))
df[scores].isna().sum()

math_score       22
reading_score    11
writing_score     5
dtype: int64

**2f.** Fill the NaN values in each score column with **that column's median**.

✅ Checkpoint: total missing values in `df` = `0`. Means: math `66.05`, reading `69.22`, writing `68.07`.

In [9]:
for col in scores:
    df[col]=df[col].fillna(df[col].median())
display(df.isna().sum().sum())
df[scores].mean().round(2)

np.int64(0)

math_score       66.05
reading_score    69.22
writing_score    68.07
dtype: float64

---
## Part 3: select data

**3a.** How many students scored **90 or more** in math?

✅ Checkpoint: `56`

In [10]:
(df['math_score']>=90).sum()

np.int64(56)

**3b.** How many **female** students **completed** the test prep course **and** scored **above 80** in reading?

💡 Hint: combine conditions with `&` and wrap each one in `( )`.

✅ Checkpoint: `77`

In [11]:
((df['gender']=='female')&(df['test_preparation_course']=='completed')&(df['reading_score']>80)).sum()

np.int64(77)

**3c.** Use `iloc` to show the first 5 rows and **only** the 3 score columns.

✅ Checkpoint: the first row is `85, 90, 92`.

In [12]:
df[['math_score','reading_score','writing_score']].iloc[0:5]

,math_score,reading_score,writing_score
0,85.0,90.0,92.0
1,55.0,47.0,44.0
2,69.0,86.0,81.0
3,68.0,68.0,67.0
4,73.0,78.0,74.0


---
## Part 4: new column and loops

**4a.** Add a column `average_score`: the mean of the 3 scores for each student, rounded to 2 decimals.

💡 Hint: `.mean(axis=1)`

✅ Checkpoint: `df['average_score'].mean()` is about `67.78`.

In [13]:
df['average_score']=df[scores].mean(axis=1).round(2)
round(df['average_score'].mean(),2)

np.float64(67.78)

**4b.** Use a **for loop** with `iterrows()` to count the students who scored **50 or more in all three** subjects.

✅ Checkpoint: `812`

In [14]:
count=0
for index,row in df.iterrows():
    if (row[scores]>=50).all():
        count+=1
count

812

**4c. Bonus:** get the same answer in **one line**, without a loop.

💡 Hint: `(df[scores] >= 50)` gives a table of True/False. Look up `.all(axis=1)`.

✅ Checkpoint: `812` again

In [15]:
(df[scores]>=50).all(axis=1).sum()

np.int64(812)

---
## Part 5: statistics

**5a.** For `math_score`, find the mean, median, std, mode, min and max.

✅ Checkpoint: mean `66.05`, median `66.0`, std `14.98`, mode `66.0`, min `0.0`, max `100.0`

In [16]:
df['math_score'].describe().round(2)

count    1000.00
mean       66.05
std        14.98
min         0.00
25%        57.00
50%        66.00
75%        76.00
max       100.00
Name: math_score, dtype: float64

**5b.** Run `describe()` on the 3 score columns.

✅ Checkpoint: the `mean` row matches the numbers from **2f**.

In [17]:
df[scores].describe().round(2)

,math_score,reading_score,writing_score
count,1000.00,1000.00,1000.00
mean,66.05,69.22,68.07
std,14.98,14.55,15.19
min,0.00,17.00,10.00
25%,57.00,60.00,58.00
50%,66.00,70.00,69.00
75%,76.00,79.00,79.00
max,100.00,100.00,100.00


---
## Part 6: groupby

**6a.** Average of each score **by gender**.

✅ Checkpoint: math female `63.44` / male `68.84`, reading female `72.6` / male `65.58`, writing female `72.47` / male `63.35`

In [18]:
df.groupby('gender')[scores].mean().round(2)

,math_score,reading_score,writing_score
gender,,,
female,63.44,72.60,72.47
male,68.84,65.58,63.35


**6b.** Average `average_score` by `test_preparation_course`.

✅ Checkpoint: completed `72.64`, none `65.07`

In [25]:
df.groupby('test_preparation_course')['average_score'].mean().round(2)

test_preparation_course
completed    72.64
none         65.07
Name: average_score, dtype: float64

**6c.** Average `average_score` by `parental_level_of_education`, **sorted from highest to lowest**.

✅ Checkpoint: top is `master's degree` with `73.51`, bottom is `high school` with `63.19`

In [26]:
df.groupby('parental_level_of_education')['average_score'].mean().round(2).sort_values(ascending=False)

parental_level_of_education
master's degree       73.51
bachelor's degree     71.84
associate's degree    69.53
some college          68.53
some high school      65.11
high school           63.19
Name: average_score, dtype: float64

**6d.** Average `average_score` grouped by **gender AND lunch** together.

✅ Checkpoint: highest is female + standard `73.13`, lowest is male + free/reduced `61.3`

In [31]:
df.groupby(['gender','lunch'])['average_score'].mean().round(2)

gender  lunch       
female  free/reduced    63.20
        standard        73.13
male    free/reduced    61.30
        standard        68.35
Name: average_score, dtype: float64

---
## Part 7: correlation

**7a.** Show the correlation matrix of the 3 scores. Which **pair** is most related?

✅ Checkpoint: reading ↔ writing `0.95`

In [33]:
df[scores].corr().round(2)

,math_score,reading_score,writing_score
math_score,1.00,0.80,0.79
reading_score,0.80,1.00,0.95
writing_score,0.79,0.95,1.00


---
## Part 8: JSON and merging

**8a.** Read `students_extra.json` into `extra`.

✅ Checkpoint: shape is `(980, 3)`. That's 20 fewer students than `df`. Keep that in mind 👀

In [34]:
extra=pd.read_json('students_extra.json')
extra.shape

(980, 3)

**8b.** Merge `df` with `extra` on `student_id` **twice**: once with `how='inner'` and once with `how='left'`. Compare the row counts. Why are they different?

✅ Checkpoint: inner `980` rows, left `1000` rows. In the left merge, `20` rows have NaN for `study_hours_per_week`.

In [35]:
inner= df.merge(extra,on='student_id',how='inner')
left= df.merge(extra,on='student_id',how='left')
print(len(left),len(inner))
print(left['study_hours_per_week'].isna().sum())

1000 980
20


**8c.** Using the **inner** merge, how do `study_hours_per_week` and `attendance_rate` correlate with `average_score`?

✅ Checkpoint: study hours `0.8`, attendance `0.72`

In [39]:
inner[['average_score','study_hours_per_week','attendance_rate']].corr().round(2)

,average_score,study_hours_per_week,attendance_rate
average_score,1.00,0.80,0.72
study_hours_per_week,0.80,1.00,0.59
attendance_rate,0.72,0.59,1.00


---
## Part 9: reshaping

**9a.** Use `melt` to turn the 3 score columns into **long format** with columns `student_id`, `gender`, `subject`, `score`.

✅ Checkpoint: shape is `(3000, 4)`. Each student now has 3 rows.

In [43]:
long=df.melt(id_vars=['student_id','gender'],value_vars=scores,var_name='subject',value_name='score')
print(long.shape)
long.head()

(3000, 4)


,student_id,gender,subject,score
0,1821,female,math_score,85.0
1,1734,male,math_score,55.0
2,1829,female,math_score,69.0
3,1570,male,math_score,68.0
4,1902,female,math_score,73.0


**9b.** Use `pivot_table` on `long`: genders as rows, subjects as columns, mean score as values.

✅ Checkpoint: **the same numbers as 6a**. Two different roads, same answer.

In [50]:
long.pivot_table(index='gender',columns='subject',values='score',aggfunc='mean').round(2)

subject,math_score,reading_score,writing_score
gender,,,
female,63.44,72.60,72.47
male,68.84,65.58,63.35


---
## Part 10: your findings

**10.** In the Markdown cell below, write **3 findings** about these students in your own words. What would you tell a teacher?

Example: *"Students who completed test prep scored about 7.5 points higher on average."*

1.students tend to score lower in math with an average of 63.44
2.students who score high in writing have a high chance to also get high score in reading cus the corr between them is (95)
3.study hours matter more than attendance for getting a higher average